# Phase 3.2 – Road Damage YOLO Model Training
This notebook handles dataset verification, model loading (YOLOv11n), training on Apple Silicon GPU (MPS), and basic inference testing.

In [ ]:
import os

dataset_path = "datasets/road_damage/RDD2022" if os.path.exists("datasets/road_damage/RDD2022") else "datasets/road_damage"

print("Train Images:", len(os.listdir(f"{dataset_path}/train/images")))
print("Train Labels:", len(os.listdir(f"{dataset_path}/train/labels")))

print("Validation Images:", len(os.listdir(f"{dataset_path}/valid/images")))
print("Validation Labels:", len(os.listdir(f"{dataset_path}/valid/labels")))

if os.path.exists(f"{dataset_path}/test/images"):
    print("Test Images:", len(os.listdir(f"{dataset_path}/test/images")))
if os.path.exists(f"{dataset_path}/test/labels"):
    print("Test Labels:", len(os.listdir(f"{dataset_path}/test/labels")))


In [ ]:
with open("datasets/road_damage/data.yaml", "r") as f:
    print(f.read())


In [ ]:
import torch
from ultralytics import YOLO

print("PyTorch Version:", torch.__version__)
print("MPS Available:", torch.backends.mps.is_available())

# Load YOLO model
model = YOLO("yolo11n.pt")

# Train model
results = model.train(
    data="datasets/road_damage/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device="mps",
    workers=2,
    project="runs",
    name="road_damage_v1"
)


In [ ]:
# Test inference on an image
model = YOLO("runs/road_damage_v1/weights/best.pt")

# Predict on test sample image if available
import glob

test_images = glob.glob("datasets/road_damage/RDD2022/test/images/*.*") or glob.glob("datasets/road_damage/test/images/*.*")
if test_images:
    sample_img = test_images[0]
    print(f"Testing prediction on: {sample_img}")
    results = model.predict(
        source=sample_img,
        conf=0.5,
        save=True
    )
else:
    print("No test images found. Provide a path to an image to test prediction.")
